In [1]:
import pandas as pd
import numpy as np

hc = pd.read_csv("data/heart_2020_cleaned.csv")
hc.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319795 entries, 0 to 319794
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   HeartDisease      319795 non-null  object 
 1   BMI               319795 non-null  float64
 2   Smoking           319795 non-null  object 
 3   AlcoholDrinking   319795 non-null  object 
 4   Stroke            319795 non-null  object 
 5   PhysicalHealth    319795 non-null  float64
 6   MentalHealth      319795 non-null  float64
 7   DiffWalking       319795 non-null  object 
 8   Sex               319795 non-null  object 
 9   AgeCategory       319795 non-null  object 
 10  Race              319795 non-null  object 
 11  Diabetic          319795 non-null  object 
 12  PhysicalActivity  319795 non-null  object 
 13  GenHealth         319795 non-null  object 
 14  SleepTime         319795 non-null  float64
 15  Asthma            319795 non-null  object 
 16  KidneyDisease     31

In [2]:
def calculate_pearson_r_py(x_data, y_data):
    """
    Обчислює коефіцієнт кореляції Пірсона (r) за "сирою" формулою.
    
    Приймає:
    x_data (list): Список значень змінної X.
    y_data (list): Список значень змінної Y.
    
    Повертає:
    float: Коефіцієнт кореляції Пірсона (r).
    """
    n = len(x_data)
    
    if n != len(y_data):
        raise ValueError("Довжина списків X та Y має збігатися.")
    if n < 2:
        return float('nan')

    # Ініціалізуємо суми
    sx, sy, sxy, sx2, sy2 = 0.0, 0.0, 0.0, 0.0, 0.0

    # Обчислюємо всі необхідні суми за один прохід
    for x_val, y_val in zip(x_data, y_data):
        sx += x_val
        sy += y_val
        sxy += x_val * y_val
        sx2 += x_val**2
        sy2 += y_val**2
    
    # Чисельник (n * Σ(xy) - Σx * Σy)
    numerator = n * sxy - sx * sy
    
    # Знаменник (з вашого коду: sqrt([n*Σx² - (Σx)²] * [n*Σy² - (Σy)²]))
    den_x = n * sx2 - sx**2
    den_y = n * sy2 - sy**2
    
    denominator = (den_x * den_y)**0.5
    
    # Уникаємо ділення на нуль
    if denominator == 0:
        return float('nan')
        
    return numerator / denominator

sleep_time = hc['SleepTime'].values.astype(float)
mental_health = hc['MentalHealth'].values.astype(float)
r_manual = calculate_pearson_r_py(hc['SleepTime'], hc['MentalHealth'])

print(f"Коефіцієнт кореляції Python: {r_manual:.4f}")

Коефіцієнт кореляції Python: -0.1197


In [3]:
sleep_time = hc['SleepTime']
mental_health = hc['MentalHealth']
r_manual = calculate_pearson_r_py(sleep_time, mental_health)

print(f"Коефіцієнт кореляції r (Ваша функція): {r_manual:.4f}")

Коефіцієнт кореляції r (Ваша функція): -0.1197


In [4]:
pearson_r = sleep_time.corr(mental_health)
print(f"Коефіцієнт кореляції Пірсона (Pandas): {pearson_r:.4f}")


Коефіцієнт кореляції Пірсона (Pandas): -0.1197


In [5]:
%timeit calculate_pearson_r_py(hc["SleepTime"].values, hc["MentalHealth"].values)


336 ms ± 63.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
import os
os.environ["CC"] = "mingw32-gcc.exe"

In [7]:
%load_ext Cython

In [8]:
%%cython
from libc.math cimport sqrt

def calculate_pearson_r_cy(double[::1] x_data, double[::1] y_data):
    """
    Обчислює коефіцієнт кореляції Пірсона (r) мовою Cython.
    
    Параметри:
        x_data: масив значень X (numpy.ndarray dtype=float64, contiguous)
        y_data: масив значень Y (numpy.ndarray dtype=float64, contiguous)
    
    Повертає:
        float: коефіцієнт кореляції Пірсона (r).
    """
    cdef Py_ssize_t n = x_data.shape[0]
    if n != y_data.shape[0]:
        raise ValueError("Довжина X та Y має збігатися.")
    if n < 2:
        return float('nan')
    
    # Локальні змінні
    cdef double sx = 0.0
    cdef double sy = 0.0
    cdef double sxy = 0.0
    cdef double sx2 = 0.0
    cdef double sy2 = 0.0
    cdef double numerator, den_x, den_y, denominator
    cdef Py_ssize_t i
    
    # Один прохід по даних
    for i in range(n):
        sx += x_data[i]
        sy += y_data[i]
        sxy += x_data[i] * y_data[i]
        sx2 += x_data[i] * x_data[i]
        sy2 += y_data[i] * y_data[i]
    
    numerator = n * sxy - sx * sy
    den_x = n * sx2 - sx * sx
    den_y = n * sy2 - sy * sy
    denominator = sqrt(den_x * den_y)
    
    if denominator == 0:
        return float('nan')
    
    return numerator / denominator


DistutilsPlatformError: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/